# Mezcla sistemática de audios (por lotes)

Con este código, se genera una mezcla sistematica de los datos de audio usando TODOS los archivos de speech disponibles.

Se usa procesamiento por lotes (batches) para mantener el uso de RAM bajo control:
en ningún momento hay más de BATCH_SIZE archivos cargados en memoria al mismo tiempo.

Total: 426 × 5 = 2,130 mezclas aproximadamente.
Split: 70% train / 15% val / 15% test


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nhattruongdev/musan-noise")

print("Path to dataset files:", path)

100%|██████████| 10.3G/10.3G [07:03<00:00, 26.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nhattruongdev/musan-noise/versions/1


In [ ]:
"""
Fase 2 — Mezcla sintética MUSAN (versión por lotes)
Proyecto C: Audio NMF (separación de fuentes)

Restricciones:
  - Todos los archivos de speech disponibles
  - 5 niveles de SNR: 0, 5, 10, 15, 20 dB
  - Split por archivo de speech: 70/15/15 (train/val/test)
  - Mismo speech aparece en los 5 niveles → comparación justa entre niveles
  - Contaminante: solo noise
  - Procesamiento por lotes para cuidar la RAM
"""
import numpy as np
import soundfile as sf
import librosa
import gc
from pathlib import Path
import random
import json
from collections import defaultdict

In [ ]:

semilla    = 42      
BATCH_SIZE = 50       #archivos de speech procesados a la vez 

random.seed(semilla)
np.random.seed(semilla)

SR         = 16000
SNR_LEVELS = [0, 5, 10, 15, 20]
SPLIT      = (0.70, 0.15, 0.15)   # train - val - test

In [ ]:
def load_audio(path, sr=SR):
    """Carga y resamplea a mono 16kHz."""
    audio, _ = librosa.load(str(path), sr=sr, mono=True)
    return audio.astype(np.float32)

def rms(x):
    """RMS con epsilon para evitar división por cero."""
    return float(np.sqrt(np.mean(x ** 2) + 1e-9))

def mix_at_snr(speech, contaminant, snr_db):
    """
    Mezcla speech + contaminant al SNR objetivo.

    Si el contaminante es más corto que el speech, se repite (tile).
    Se toma un offset aleatorio para no usar siempre el inicio.
    La mezcla se normaliza al final para evitar clipping.

    Retorna: (mixed, alpha, snr_verificado)
    """
    if len(contaminant) < len(speech):
        reps = int(np.ceil(len(speech) / len(contaminant)))
        contaminant = np.tile(contaminant, reps)

    max_start = len(contaminant) - len(speech)
    start = random.randint(0, max_start) if max_start > 0 else 0
    contaminant = contaminant[start : start + len(speech)]

    alpha = rms(speech) / (rms(contaminant) * 10 ** (snr_db / 20.0))
    mixed = speech + alpha * contaminant

    peak = np.max(np.abs(mixed))
    if peak > 0:
        mixed = mixed / peak

    snr_real = 20 * np.log10(rms(speech) / (rms(alpha * contaminant) + 1e-9))

    return mixed.astype(np.float32), float(alpha), round(float(snr_real), 2)

def cycle_sample(file_list, n):
    """
    Devuelve n paths sin repetición inmediata excesiva.
    Si n > len(file_list): ciclos completos + muestra del resto.
    Siempre hace shuffle para variar el orden entre llamadas.
    """
    if len(file_list) == 0:
        raise ValueError("La lista de archivos está vacía.")
    full_cycles = n // len(file_list)
    remainder   = n %  len(file_list)
    pool = file_list * full_cycles
    if remainder > 0:
        pool += random.sample(file_list, remainder)
    random.shuffle(pool)
    return pool

def split_speech(speech_files, split=(0.70, 0.15, 0.15)):
    """
    Divide los archivos de speech en train/val/test.
    El split es FIJO para todos los niveles de SNR.
    """
    files = speech_files.copy()
    random.shuffle(files)
    n = len(files)
    n_train = int(n * split[0])
    n_val   = int(n * split[1])
    return (
        files[:n_train],
        files[n_train : n_train + n_val],
        files[n_train + n_val :]
    )

def process_batch(speech_batch, noise_files, subset_name, snr, out_snr_dir,
                  batch_offset, metadata, snr_errors):
    """
    Procesa un lote de archivos de speech para un nivel de SNR dado.
    Carga, mezcla, escribe al disco y libera memoria al terminar.
    """
    noise_pool = cycle_sample(noise_files, len(speech_batch))

    for i, (sp_path, noise_path) in enumerate(zip(speech_batch, noise_pool)):
        try:
            speech      = load_audio(sp_path)
            contaminant = load_audio(noise_path)
            mixed, alpha, snr_real = mix_at_snr(speech, contaminant, snr)

            out_path = out_snr_dir / f"mix_{batch_offset + i:04d}.wav"
            sf.write(str(out_path), mixed, SR)

            metadata.append({
                "file"            : str(out_path),
                "subset"          : subset_name,
                "speech_src"      : sp_path,
                "contaminant_src" : noise_path,
                "contaminant_type": "noise",
                "snr_target_db"   : snr,
                "snr_real_db"     : snr_real,
                "alpha"           : alpha,
            })

            error = abs(snr_real - snr)
            if error > 1.0:
                snr_errors[snr].append({
                    "file"    : str(out_path),
                    "snr_real": snr_real,
                    "error_db": round(error, 2)
                })

            # Liberar memoria explícitamente después de cada mezcla
            del speech, contaminant, mixed

        except Exception as e:
            print(f"  [ERROR] {sp_path} | {noise_path} → {e}")

    # Forzar liberación de RAM al terminar el lote
    gc.collect()



def build_dataset(speech_dir, noise_dir, out_dir, batch_size=BATCH_SIZE):
    """
    Genera el dataset de mezclas sintéticas usando solo speech + noise.
    Procesa en lotes de batch_size para mantener la RAM bajo control.

    Estructura de salida:
        out_dir/
          train/snr_Xdb/mix_XXXX.wav
          val/snr_Xdb/mix_XXXX.wav
          test/snr_Xdb/mix_XXXX.wav
          metadata.json
          split_info.json
    """
    out_dir = Path(out_dir)

    speech_files = sorted(Path(speech_dir).rglob("*.wav"))
    noise_files  = sorted(Path(noise_dir).rglob("*.wav"))

    speech_files = [str(p) for p in speech_files]
    noise_files  = [str(p) for p in noise_files]

    print(f"Speech disponible : {len(speech_files):>4} archivos")
    print(f"Noise disponible  : {len(noise_files):>4} archivos")
    print(f"Batch size        : {batch_size} archivos por lote")
    print()

    if len(speech_files) == 0:
        raise FileNotFoundError(f"No se encontraron .wav en {speech_dir}")
    if len(noise_files) == 0:
        raise FileNotFoundError(f"No se encontraron .wav en {noise_dir}")

    # Split fijo de speech
    train_sp, val_sp, test_sp = split_speech(speech_files, SPLIT)
    split_info = {
        "seed"        : semilla,
        "total_speech": len(speech_files),
        "train"       : len(train_sp),
        "val"         : len(val_sp),
        "test"        : len(test_sp),
        "snr_levels"  : SNR_LEVELS,
        "contaminant" : "noise_only",
        "batch_size"  : batch_size,
    }
    print(f"Split de speech → train: {len(train_sp)} | val: {len(val_sp)} | test: {len(test_sp)}")
    print()

    metadata   = []
    snr_errors = defaultdict(list)

    subsets = [("train", train_sp), ("val", val_sp), ("test", test_sp)]

    for subset_name, speech_subset in subsets:
        for snr in SNR_LEVELS:
            out_snr_dir = out_dir / subset_name / f"snr_{snr}db"
            out_snr_dir.mkdir(parents=True, exist_ok=True)

            # Dividir el subset en lotes
            batches = [
                speech_subset[i : i + batch_size]
                for i in range(0, len(speech_subset), batch_size)
            ]

            for b_idx, batch in enumerate(batches):
                batch_offset = b_idx * batch_size
                process_batch(
                    batch, noise_files, subset_name, snr,
                    out_snr_dir, batch_offset, metadata, snr_errors
                )
                print(f"  [{subset_name:5s}] SNR {snr:>2} dB — lote {b_idx + 1}/{len(batches)} completado ({len(batch)} mezclas)")

        print()

    meta_path = out_dir / "metadata.json"
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    split_path = out_dir / "split_info.json"
    with open(split_path, "w") as f:
        json.dump(split_info, f, indent=2)

    
    total    = len(metadata)
    snr_warn = sum(len(v) for v in snr_errors.values())

    print("=" * 52)
    print(f"  Total de mezclas generadas : {total}")
    print(f"  Contaminante               : noise (100%)")
    print(f"  Mezclas con error SNR >1dB : {snr_warn}")
    print(f"  Metadata guardado en       : {meta_path}")
    print("=" * 52)

    return metadata, split_info

In [ ]:
import os
os.chdir(path)

build_dataset(
    speech_dir = "musan/speech",
    noise_dir  = "musan/noise",
    out_dir    = "mixed_audio"
)

Speech disponible :  426 archivos
Noise disponible  :  930 archivos
Batch size        : 50 archivos por lote

Split de speech → train: 298 | val: 63 | test: 65

  [train] SNR  0 dB — lote 1/6 completado (50 mezclas)
  [train] SNR  0 dB — lote 2/6 completado (50 mezclas)
  [train] SNR  0 dB — lote 3/6 completado (50 mezclas)
  [train] SNR  0 dB — lote 4/6 completado (50 mezclas)
  [train] SNR  0 dB — lote 5/6 completado (50 mezclas)
  [train] SNR  0 dB — lote 6/6 completado (48 mezclas)
  [train] SNR  5 dB — lote 1/6 completado (50 mezclas)
  [train] SNR  5 dB — lote 2/6 completado (50 mezclas)
  [train] SNR  5 dB — lote 3/6 completado (50 mezclas)
  [train] SNR  5 dB — lote 4/6 completado (50 mezclas)
  [train] SNR  5 dB — lote 5/6 completado (50 mezclas)
  [train] SNR  5 dB — lote 6/6 completado (48 mezclas)
  [train] SNR 10 dB — lote 1/6 completado (50 mezclas)
  [train] SNR 10 dB — lote 2/6 completado (50 mezclas)
  [train] SNR 10 dB — lote 3/6 completado (50 mezclas)
  [train] SNR 

([{'file': 'mixed_audio/train/snr_0db/mix_0000.wav',
   'subset': 'train',
   'speech_src': 'musan/speech/us-gov/speech-us-gov-0239.wav',
   'contaminant_src': 'musan/noise/free-sound/noise-free-sound-0040.wav',
   'contaminant_type': 'noise',
   'snr_target_db': 0,
   'snr_real_db': -0.0,
   'alpha': 0.3571265895395356},
  {'file': 'mixed_audio/train/snr_0db/mix_0001.wav',
   'subset': 'train',
   'speech_src': 'musan/speech/librivox/speech-librivox-0144.wav',
   'contaminant_src': 'musan/noise/free-sound/noise-free-sound-0798.wav',
   'contaminant_type': 'noise',
   'snr_target_db': 0,
   'snr_real_db': -0.0,
   'alpha': 0.44858466528905283},
  {'file': 'mixed_audio/train/snr_0db/mix_0002.wav',
   'subset': 'train',
   'speech_src': 'musan/speech/us-gov/speech-us-gov-0080.wav',
   'contaminant_src': 'musan/noise/free-sound/noise-free-sound-0181.wav',
   'contaminant_type': 'noise',
   'snr_target_db': 0,
   'snr_real_db': 0.0,
   'alpha': 3.4997204486831066},
  {'file': 'mixed_audio/